# 02 — QLoRA Fine-Tuning (Kaggle GPU)

⚠️ **This notebook needs a Kaggle GPU (T4 or better).** It will not run usefully on a CPU-only machine.

What happens here (Stage B supervised fine-tuning):

- Base model: **Qwen/Qwen2.5-1.5B-Instruct** (see `model_name` in `config.yaml`)
- Method: **QLoRA** — 4-bit quantized base weights + trainable LoRA adapters
  (`lora_rank: 16`, `lora_alpha: 32`, `lora_dropout: 0.05`, `learning_rate: 0.0002`,
  `epochs: 3`, `batch_size: 4` × `gradient_accumulation: 4`)
- Stage A (continued pre-training) is **disabled** in this config (`stage_a_enabled: false`)
- Output: LoRA adapter saved to `models/adapters/agri-climate-qlora` (`adapter_dir`)

### Kaggle setup

1. Create a Kaggle notebook with **GPU T4 x2** (or P100) accelerator enabled.
2. Add this repo (upload as a dataset or `git clone` it into `/kaggle/working`).
3. Run the cells below in order.

## 1. Install dependencies

Heavy ML packages come from `requirements.txt`. (Skip on Kaggle if you use an image that already has torch/transformers.)

In [ ]:
# On Kaggle, cd to the repo first if needed, e.g.:
# %cd /kaggle/working/agriculture-climate-slm
%pip install -q -r requirements.txt

## 2. Confirm a GPU is available

QLoRA training on CPU is not practical — stop here if no CUDA device is found.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("NO GPU FOUND — enable a Kaggle GPU accelerator before continuing.")

## 3. Load the training configuration

All hyper-parameters are read from `config.yaml` (single source of truth).

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists() and (ROOT.parent / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.config_loader import load_config

cfg = load_config(str(ROOT / "config.yaml"))
print(json.dumps(
    {k: cfg[k] for k in (
        "model_name", "max_seq_length", "learning_rate", "batch_size",
        "gradient_accumulation", "epochs", "lora_rank", "lora_alpha",
        "lora_dropout", "quantization", "train_path", "validation_path", "adapter_dir",
    )},
    indent=2,
))

## 4. Launch QLoRA SFT

`python -m src.training.train` runs the full Stage B loop (4-bit quantize → attach LoRA →
supervised fine-tune on `data/train/train.jsonl` → validate → save adapter). This is the long step;
expect it to take on the order of tens of minutes on a T4, depending on dataset size.

In [ ]:
!python -m src.training.train

## 5. Verify the adapter was saved

The adapter directory (`models/adapters/agri-climate-qlora`) should now contain the LoRA weights
and config. Download this directory from Kaggle — the evaluation and demo stages use it.

In [ ]:
adapter = ROOT / cfg["adapter_dir"]
print("adapter dir:", adapter, "exists:", adapter.exists())
if adapter.exists():
    for p in sorted(adapter.iterdir()):
        print(" ", p.name)